# New Solar Radiation Derivative Products

***

## Comment on the Existing SoRad / SoREd Products

The paper's SoRad contract (Eq. 18) is essentially a **vanilla put on daily GHI**
with a single maturity date $T$:

$$
\Gamma(R_T, K) = \text{tick} \times (K - R_T)\,\mathbf{1}_{R_T < K}
$$

The SoRadIDX index simply sums $N$ such dailies into a seasonal basket.
This design has three **practical limitations**:

1. **Single-day exposure**: A PV operator's revenue risk is cumulative over a period
   (e.g. a month or quarter), not just one day. A single bad day is not hedgeable
   in isolation - the relevant risk is sustained low radiation.
2. **No path dependency**: The contract pays regardless of whether radiation was
   consistently low or just volatile around the strike. A solar producer exposed
   to a long cloudy spell needs a different instrument.
3. **Fixed strike**: The seasonal mean $\bar{Y}_T$ is used as the natural strike,
   but producers may want protection specifically against *cumulative shortfalls*
   below a seasonal budget, not a daily threshold.

## Product 1: Asian SoRad (ASoRad) — Payoff on Average Radiation

**Motivation:** mirrors the actual revenue of a PV plant, which depends on the
*total* radiation over a settlement period $[T_1, T_2]$.

**Payoff:**

$$
\Gamma^{\text{Asian}}(\bar{R}, K) = \text{tick} \times \left(K - \bar{R}_{[T_1,T_2]}\right)_+
$$

where the arithmetic average is:

$$
\bar{R}_{[T_1,T_2]} = \frac{1}{T_2 - T_1} \sum_{n=T_1}^{T_2} R_n
$$

#### 1.1 Key Technical Details

- **Distribution of $\bar{R}$**: In Y-space, each $Y_n$ is a Gaussian mixture.
  The average $\bar{Y}$ is also approximately Gaussian (CLT) with mean and variance:

$$
\mathbb{E}[\bar{Y}] = \frac{1}{N}\sum_{n=T_1}^{T_2} M_{Y_i}(t,n), \qquad
\text{Var}[\bar{Y}] = \frac{1}{N^2}\sum_{n,m} \text{Cov}(Y_n, Y_m)
$$

  The covariance between $Y_n$ and $Y_m$ under the OU process is:

$$
\text{Cov}(Y_n, Y_m) = e^{-\theta|n-m|} S^2_Y\!\left(t,\, \min(n,m)\right)
$$

- **Pricing**: With the Gaussian approximation for $\bar{Y}$, back-transform via
  Eq. 2 and price analogously to Eq. 19, replacing single-maturity moments with
  the average-period moments.

- **Advantage over SoRad**: Reduces daily noise and lowers basis risk for
  monthly or quarterly settlement contracts.

#### 1.2 How Appendix B Changes for ASoRad

Appendix B currently derives two objects for the single-maturity SoRad/SoREd:
the conditional expectation $M_{Y_i}(t,T)$ (§B.1) and the conditional variance
$S^2_{Y_i}(t,T)$ (§B.2) of $Y_T$ given state $B_T = i$. For ASoRad,
a new subsection is required deriving moments of the **average** $\bar{Y}$.

**Conditional mean of the average.** For each day $n \in [T_1, T_2]$, the existing
formula B.7 already gives:

$$
M_{Y_i}(t, n) = \bar{Y}_n + (Y_t - \bar{Y}_t)e^{-\theta(n-t)}
+ \mathbb{E}[\Lambda(t,n) \mid B_n = i]
$$

The average mean is then:

$$
M_{\bar{Y}_i}(t) = \frac{1}{N}\sum_{n=T_1}^{T_2} M_{Y_i}(t,n)
$$

**Conditional variance of the average (new derivation).** Using the OU solution
(Eq. 10 in the paper), for $n < m$:

$$
\text{Cov}(Y_n, Y_m \mid B_n = i, B_m = j) = e^{-\theta(m-n)} S^2_{Y_i}(t, n)
$$

where $S^2_{Y_i}(t,n)$ is the existing B.16 formula evaluated at the earlier
date $n$. The **average variance** then becomes:

$$
S^2_{\bar{Y}_i}(t) = \frac{1}{N^2} \sum_{n=T_1}^{T_2} S^2_{Y_i}(t,n)
+ \frac{2}{N^2} \sum_{T_1 \le n < m \le T_2} e^{-\theta(m-n)} S^2_{Y_i}(t,n)
$$

The double sum is computable in closed form using the geometric series structure
of the exponential decay in $\theta$.

**Change of measure.** The §B.1.2 and §B.2.2 logic carries over directly:
$\tilde{\mathbb{Q}}$ shifts the mean of $\bar{Y}$ by the aggregate standard
deviation $S_{\bar{Y}_i}$, while the variance under $\tilde{\mathbb{Q}}$
picks up the corresponding additional term from the market price of risk.

#### 1.3 How Appendix C Changes for ASoRad

**C.1 — Pricing: New Proposition C1-Asian.** The existing Proposition C1 prices
the contract using the single-maturity integral. For ASoRad, a new proposition is
needed with the same proof structure but using the average-period moments.

**C.2 — Greeks: Extended Sensitivity Analysis.** The structure of C.6–C.8 is
preserved but the single-maturity terms are replaced by their average-period
counterparts. This produces a smoother, smaller delta compared with single-day
SoRad.

**C.3 — Mean-Variance Hedging: Modified Optimal Holdings.** The correction factor
and residual variance are updated by replacing single-maturity moments with the
average-period moments across $[T_1, T_2]$.

## Product 2: Barrier SoRad (BSoRad) — Knock-In on Consecutive Low-Radiation Days

**Motivation:** protection specifically against *sustained* cloudy spells,
the scenario most damaging to grid-connected solar producers.

#### 2a. Down-and-In SoRad (Knock-In)

The contract **activates** only if radiation falls below a barrier $H < K$
for at least $\tau^*$ consecutive days during $[T_1, T_2]$:

$$
\Gamma^{\text{DI}}(R_T, K) = \text{tick} \times (K - R_T)_+ \cdot
\mathbf{1}_{\left\{\exists\;\text{run of } \tau^* \text{ days with } R_n < H\right\}}
$$

#### 2b. Cumulative Barrier SoRad

Activates if the *cumulative shortfall* over the period exceeds a threshold $B$:

$$
\Gamma^{\text{CB}} = \text{tick} \times (K - R_T)_+ \cdot
\mathbf{1}_{\left\{\sum_{n=T_1}^{T_2}(H - R_n)_+ \;>\; B\right\}}
$$

#### 2.1 Key Technical Details

- **Pricing approach**: No closed-form exists. Price via **Monte Carlo simulation**
  of the discretized OU process.
- **Bernoulli regime switching**: The barrier condition naturally captures the
  clustering of cloudy days through persistence in the OU mean.
- **Greeks**: Delta with respect to $R_t$ can be computed via
  pathwise differentiation or the likelihood ratio method in the Monte Carlo.

#### 2.2 How Appendix B Changes for BSoRad

The barrier products do **not require any modification to Appendix B** because
Monte Carlo simulation bypasses the need for closed-form conditional moments.

#### 2.3 How Appendix C Changes for BSoRad

**C.1 — Pricing: Monte Carlo Remark.** Add a short remark noting that barrier
products are priced by Monte Carlo using the OU discretization.

**C.2 — Greeks: Likelihood Ratio Estimator.** The barrier discontinuity means
pathwise differentiation of the indicator is not valid directly, so the Likelihood
Ratio estimator should be used instead.

**C.3 — Mean-Variance Hedging: Numerical Hedge Ratio.** No closed-form optimal
holding formula exists; the hedge ratio is computed numerically.

***

## Summary Comparison of All Products

| Product | Payoff driver | Path dependent | Pricing method | Appendix B change | Appendix C change | Natural use case |
|---|---|---|---|---|---|---|
| **SoRad** (existing) | $R_T$ single day | No | Semi-analytical Eq. 19 | None | None | Daily spot exposure |
| **SoRadIDX** (existing) | Sum of daily puts | No | Sum of Eq. 19 | None | None | Seasonal basket |
| **ASoRad** (proposed) | $\bar{R}_{[T_1,T_2]}$ average | Weakly | Semi-analytical (CLT approx.) | New average-moment derivation | New Proposition C1-Asian; modified hedge ratio | Monthly/quarterly settlement |
| **BSoRad DI** (proposed) | $R_T$ + run barrier | Yes | Monte Carlo | Unchanged (used for MC init) | MC remark; LR delta estimator | Protection against sustained cloudy spells |
| **BSoRad CB** (proposed) | $R_T$ + cumulative shortfall | Yes | Monte Carlo | Unchanged | Same as BSoRad DI | Budget shortfall insurance |

The Asian variant fits naturally within the paper's analytical framework since
the Gaussian mixture structure is preserved under averaging. The barrier variants
require simulation but are directly implementable using the OU discretization
already built in the notebook.